# SECAAE Analista V2
Protótipo completo para Google Colab: Google Drive → DuckDB → Gemini → Streamlit.

In [ ]:
!pip -q install -r /content/secaae_analista_v2/requirements.txt


In [ ]:
from google.colab import auth
auth.authenticate_user()
print('Google Drive autenticado.')


In [ ]:
import os, sys
PROJECT = '/content/secaae_analista_v2'
sys.path.insert(0, PROJECT)

# Informe os valores abaixo apenas nesta sessão do Colab.
os.environ['GEMINI_API_KEY'] = 'COLE_SUA_CHAVE_GEMINI_AQUI'
os.environ['GEMINI_MODEL'] = 'gemini-3.6-flash'
os.environ['DRIVE_FOLDER_ID'] = 'COLE_O_ID_DA_PASTA_DASHBOARD_AQUI'
os.environ['DB_PATH'] = f'{PROJECT}/data/analytics.duckdb'

print('Configuração carregada.')


In [ ]:
from data.database import Database
from data.ingestion import ingest_drive

db = Database(os.environ['DB_PATH'])
stats = ingest_drive(
    db=db,
    folder_id=os.environ['DRIVE_FOLDER_ID'],
    folder_name='Dashboard',
    recursive=False,
    progress_callback=print,
)
stats


In [ ]:
print(db.schema_context())


In [ ]:
!cd /content/secaae_analista_v2 && streamlit run app.py --server.address 0.0.0.0 --server.port 8501 > /tmp/streamlit.log 2>&1 &
!sleep 5
!cat /tmp/streamlit.log | tail -30


## Opcional: ngrok
Instale e configure seu token em `NGROK_AUTHTOKEN` sem gravá-lo no projeto.

In [ ]:
!pip -q install pyngrok

import os
from pyngrok import ngrok

token = os.getenv('NGROK_AUTHTOKEN', '')
if not token:
    print('Defina NGROK_AUTHTOKEN antes de executar esta célula.')
else:
    ngrok.set_auth_token(token)
    tunnel = ngrok.connect(8501)
    print('URL pública:', tunnel.public_url)
